In [ ]:
#@title 1. Parâmetros da Execução (Usuário)
repo_url = "https://github.com/coelhoenilda/01-phd-ds-eto.git" #@param {type:"string"}
git_ref = "main" #@param {type:"string"}
config_file = "config/cycle_01_baseline.yaml" #@param {type:"string"}
run_id_override = "r01" #@param {type:"string"}
usar_drive = False #@param {type:"boolean"}

In [ ]:
#@title 2. Preparar Ambiente, Repositório e Quarto
import os
import subprocess
from pathlib import Path

repo_dir = Path("/content/drive/01-phd-ds-eto")

if usar_drive:
    from google.colab import drive
    drive.mount("/content/drive")

# Utiliza a URL autenticada caso o token tenha sido preenchido na célula anterior
target_url = authenticated_url if 'authenticated_url' in locals() else repo_url

if not repo_dir.exists():
    subprocess.run(["git", "clone", "--branch", git_ref, target_url, str(repo_dir)], check=True)
else:
    subprocess.run(["git", "-C", str(repo_dir), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(repo_dir), "checkout", git_ref], check=True)
    subprocess.run(["git", "-C", str(repo_dir), "pull", "origin", git_ref], check=True)

os.chdir(repo_dir)
if str(repo_dir) not in os.sys.path:
    os.sys.path.insert(0, str(repo_dir))

# Instala Quarto caso ainda não esteja disponível no ambiente do Colab
if subprocess.run(["which", "quarto"], capture_output=True).returncode != 0:
    quarto_deb = "/tmp/quarto-linux-amd64.deb"
    subprocess.run([
        "wget", "-q",
        "https://quarto.org/download/latest/quarto-linux-amd64.deb",
        "-O", quarto_deb
    ], check=True)
    subprocess.run(["dpkg", "-i", quarto_deb], check=True)

print("Diretório ativo:", Path.cwd())
subprocess.run(["quarto", "--version"], check=True)

In [ ]:
import os
from pathlib import Path

# Lista o conteúdo do diretório atual (/content) para checar se a pasta foi criada
print("Diretório atual:", Path.cwd())
print("Conteúdo do diretório:", os.listdir('.'))

In [ ]:
#@title 3. Carregar Configuração e Criar Pasta da Rodada
import json
from datetime import datetime
import yaml
import pandas as pd

config_path = Path(config_file)
cfg = yaml.safe_load(config_path.read_text(encoding="utf-8"))

if run_id_override.strip():
    cfg["experiment"]["run_id"] = run_id_override.strip()

cycle_id = cfg["experiment"]["cycle_id"]
run_id = cfg["experiment"]["run_id"]
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

run_dir = Path(cfg["outputs"]["base_runs_dir"]) / f"{cycle_id}_{run_id}_{timestamp}"
run_dir.mkdir(parents=True, exist_ok=True)

# Salva snapshot da configuração utilizada nesta execução
(run_dir / "config_snapshot.yaml").write_text(
    yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True),
    encoding="utf-8"
)

print("Pasta da rodada criada em:", run_dir)

In [ ]:
#@title 4. Auditoria de Qualidade dos Dados (Quality Gate)
from src.quality_check import evaluate_data_quality, QualityGateError

# Experimento config
input_csv = Path(cfg["data"]["input_path"])
df = pd.read_csv(input_csv, sep='\t')

try:
    quality_report = evaluate_data_quality(df, cfg)
except QualityGateError as exc:
    # Mesmo em caso de reprovação, tenta extrair/salvar relatório se possível
    print("FALHA NO QUALITY GATE:", exc)
    raise
finally:
    if "quality_report" in locals():
        (run_dir / "quality_report.json").write_text(
            json.dumps(quality_report, indent=2, ensure_ascii=False),
            encoding="utf-8"
        )

print("Status de qualidade:", quality_report["overall_status"])

In [ ]:
#@title 5. Treino Baseline e Registro de Métricas
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Ajustado de cfg["data"] para cfg["experiment"]
date_col = cfg["data"]["date_column"]
target_col = cfg["experiment"]["target_variable"]
seed = int(cfg["experiment"].get("random_seed", 42))

df_model = df.copy()
df_model[date_col] = pd.to_datetime(df_model[date_col], errors="coerce")
df_model = df_model.sort_values(date_col).dropna(subset=[target_col]).reset_index(drop=True)

# Exemplo mínimo de features numéricas (exclui data e alvo)
feature_cols = [
    c for c in df_model.columns
    if c not in [date_col, target_col] and pd.api.types.is_numeric_dtype(df_model[c])
]
df_model = df_model.dropna(subset=feature_cols).reset_index(drop=True)

# Corte cronológico simples (80% treino / 20% teste) para o MVP
split_idx = int(len(df_model) * 0.8)
X_train, X_test = df_model.loc[:split_idx-1, feature_cols], df_model.loc[split_idx:, feature_cols]
y_train, y_test = df_model.loc[:split_idx-1, target_col], df_model.loc[split_idx:, target_col]

model = RandomForestRegressor(n_estimators=100, random_state=seed, n_jobs=-1)
model.fit(X_train, y_train)
preds = model.predict(X_test)

rmse = float(np.sqrt(mean_squared_error(y_test, preds)))
mae = float(mean_absolute_error(y_test, preds))
r2 = float(r2_score(y_test, preds))

metrics = {
    "cycle_id": cycle_id,
    "run_id": run_id,
    "n_train": int(len(X_train)),
    "n_test": int(len(X_test)),
    "features_used": feature_cols,
    "rmse": round(rmse, 4),
    "mae": round(mae, 4),
    "r2": round(r2, 4),
}

(run_dir / "metrics.json").write_text(
    json.dumps(metrics, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("Métricas salvas:", metrics)

In [ ]:
#@title 6. Gerar Relatório com Quarto
import subprocess
import os

qmd_file = "thesis/cap_resultados_ciclo1.qmd"
output_dir_rel = run_dir / "quarto_report"
output_dir_rel.mkdir(parents=True, exist_ok=True)

# Caminho relativo a partir de 'thesis/' para 'run_dir'
run_dir_from_thesis = os.path.relpath(run_dir, start="thesis")

cmd = [
    "quarto", "render", qmd_file,
    "--to", "html",
    "--output-dir", str(output_dir_rel.resolve()),
    "-P", f"run_dir={run_dir_from_thesis}"
]

try:
    # Executa capturando a saída para podermos diagnosticar a causa em caso de falha
    result = subprocess.run(cmd, capture_output=True, text=True, check=True)
    print("Relatório gerado em:", output_dir_rel.resolve())
except subprocess.CalledProcessError as exc:
    print("=== ERRO DE EXECUÇÃO DO QUARTO ===")
    print("Código de Saída:", exc.returncode)
    print("\n--- LOG DE SAÍDA PADRÃO (STDOUT) ---")
    print(exc.stdout)
    print("\n--- LOG DE ERRO (STDERR) ---")
    print(exc.stderr)
    raise